In [ ]:
# pip install pandas numpy matplotlib seaborn scikit-learn plotly yellowbrick xgboost nbformat 
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, RandomForestRegressor, BaggingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report ,mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')


<h2>Loading the Dataset</h2>

In [ ]:
df=pd.read_csv('/kaggle/input/heart-disease-cleveland/Heart_disease_cleveland_new.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

<h3>Checking for any missing values</h3>

In [ ]:
missing_vals = df.isnull().sum()
missing_pct = (missing_vals / len(df)) * 100
missing_df = pd.DataFrame({'Missing Values': missing_vals, 'Percentage': missing_pct})
print(missing_df[missing_df['Missing Values'] > 0])

plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"\nNumber of duplicate records: {duplicate_count}")

<h3>Plotting the pair plot</h3>

In [ ]:
sns.pairplot(df, hue = 'target')

<h2>Handling and visualising the numerical features</h2>

In [ ]:
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca']

In [ ]:
for col in numerical_cols:
    print(f"\nValue counts for {col}:\n{df[col].value_counts()}")

In [ ]:
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols):
    plt.subplot(2, 3, i + 1)
    sns.histplot(df[col], kde=True, bins=30, color='mediumseagreen')
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols):
    plt.subplot(2, 3, i + 1)
    sns.violinplot(x=df[col], color='orchid')
    plt.title(f'Violin Plot of {col}')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap='RdBu_r', square=True)
plt.title("Correlation Matrix of Numerical Features")
plt.show()

<h2>Analyzing Outliers</h2>

In [ ]:
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols):
    plt.subplot(3, 3, i + 1)
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot of {col}')
plt.tight_layout()
plt.show()

<h3> Outlier Analysis: Interpretation of Boxplots </h3>

Each boxplot reveals the central distribution and potential outliers in a numerical feature. Here's a breakdown:

1. Age
Observation: No significant outliers detected.

Distribution: Fairly symmetric and centered around 55–60 years.

Interpretation: Age is well-distributed and does not require special handling.

2. Resting Blood Pressure (trestbps)
Observation: A few moderate outliers above 160 mmHg.

Interpretation:

While high blood pressure (e.g., >180) may seem like an outlier statistically, such values can be clinically valid.

These should not be dropped blindly; instead, flagged for review with domain knowledge.

3. Serum Cholesterol (chol)
Observation: Several significant outliers above 400 mg/dl, even up to ~560.

Interpretation:

These may indicate hypercholesterolemia or erroneous values.

Should be verified, and depending on context, either transformed (e.g., log scale) or winsorized.

4. Maximum Heart Rate Achieved (thalach)
Observation: A mild outlier below 90 bpm.

Interpretation:

This may reflect a patient unable to exert effort due to heart disease.

Could be an important indicator — retain it, but analyze its relationship to the target variable later.

5. ST Depression (oldpeak)
Observation: A few high outliers above 4.0.

Interpretation:

Such high depression values are unusual but possible in extreme ischemic cases.

Flag and consider normalization or log transformation before modeling.

6. Number of Major Vessels Colored by Fluoroscopy (ca)
Observation: Outlier seen at value 3, but it's within the expected range (0–3).

Interpretation:

This is not a true outlier. It just appears infrequent, likely because most patients have fewer vessels colored.

No corrective action needed, but consider class imbalance when analyzing this feature.

In [ ]:
df[numerical_cols].skew()

<h2>Handling and visualizing categorical features</h2>

In [ ]:
for col in categorical_cols:
    print(f"\nValue counts for {col}:\n{df[col].value_counts()}")

In [ ]:
plt.figure(figsize=(18, 12))
for i, col in enumerate(categorical_cols):
    plt.subplot(3, 3, i + 1)
    sns.countplot(x=col, data=df, palette='viridis')
    plt.title(f'Count Plot of {col}')
plt.tight_layout()
plt.show()

<h2>Handling and visualizing the Target features "Num" that represents the heart disease</h2>

In [ ]:
df[numerical_cols + ['target']].corr()['target'].sort_values(ascending=False)

In [ ]:
target_counts = df['target'].value_counts().sort_index()
print(target_counts)
target_percent = (target_counts / target_counts.sum()) * 100
print(target_percent)

In [ ]:
plt.figure(figsize=(6,4))
ax = sns.countplot(x='target', data=df, palette='pastel')

for p in ax.patches:
    count = int(p.get_height())
    ax.annotate(f'{count}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=9)

plt.title('Heart Disease Classification')
plt.xlabel('0 = No Disease, Others = Disease')
plt.ylabel('Count')
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
ax = sns.countplot(x='target', data=df, palette='pastel')

for p in ax.patches:
    count = int(p.get_height())
    ax.annotate(f'{count}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=9)

plt.title('Binary Classification of Heart Disease')
plt.xlabel('0 = No Disease, 1 = Disease')
plt.ylabel('Count')
plt.show()

# Print counts separately
print("Target Class Counts:\n", df['target'].value_counts())
print("\nTarget Class Percentages:\n", df['target'].value_counts(normalize=True) * 100)


In [ ]:
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols):
    plt.subplot(3, 3, i+1)
    sns.violinplot(data=df, x='target', y=col, palette='pastel')
    plt.title(f'{col} by Heart Disease Presence')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    ct = pd.crosstab(df[col], df['target'], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], colormap='Set2', edgecolor='black')
    axes[i].set_title(f'Target Distribution by {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Proportion')
    axes[i].legend(title='Heart Disease', loc='best')

for j in range(len(categorical_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.suptitle('Categorical Features vs Heart Disease (Target)', fontsize=16, y=1.02)
plt.show()

<h3>T-Test for Numerical Columns</h3>

In [ ]:
from scipy.stats import ttest_ind

print("T-tests for numerical columns:\n")
for col in numerical_cols:
    group0 = df[df['target'] == 0][col]
    group1 = df[df['target'] == 1][col]
    stat, p = ttest_ind(group0, group1, nan_policy='omit')
    print(f"{col}: p-value = {p:.4f} {'(Significant)' if p < 0.05 else '(Not significant)'}")

<h3>Chi-sqaure Test for Catgorical Columns</h3>

In [ ]:
from scipy.stats import chi2_contingency
print("Chi-square test for categorical variables:\n")
for col in categorical_cols:
    table = pd.crosstab(df[col], df['target'])
    stat, p, _, _ = chi2_contingency(table)
    print(f"{col}: p-value = {p:.4f} {'(Significant)' if p < 0.05 else '(Not significant)'}")

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[numerical_cols + ['target']].corr(), annot=True, fmt=".2f", square=True)
plt.title("Correlation Matrix")
plt.show()

<h2>Analyzing the age column</h2>

In [ ]:
df['age'].describe()

In [ ]:
sns.histplot(df['age'], bins=20, kde=True)

plt.axvline(df['age'].mean(), color='red', linestyle='--', label=f"Mean: {df['age'].mean():.2f}")
plt.axvline(df['age'].median(), color='green', linestyle='--', label=f"Median: {df['age'].median():.2f}")
plt.axvline(df['age'].mode()[0], color='blue', linestyle='--', label=f"Mode: {df['age'].mode()[0]}")

plt.legend()
plt.title("Distribution of Age with Mean, Median, and Mode")
plt.xlabel("Age")
plt.ylabel("Frequency")

plt.show()


The age column distribution seems to be normaly distributed because we can clearly see the bill curve.

In [ ]:
fig = px.histogram(data_frame=df, x='age')
fig.show()

print ("Mean of the dataset: ",df['age'].mean())
print ("Median of the dataset: ",df['age'].median())
print ("Mode of the dataset: ",df['age'].agg(pd.Series.mode))

Lets explore the gender base distribution of the dataset for age column

In [ ]:
sns.boxplot(x='target', y='age', data=df, palette='Set2')
plt.title('Age vs Heart Disease')

In [ ]:
from scipy.stats import ttest_ind
ttest_ind(df[df['target']==0]['age'], df[df['target']==1]['age'], nan_policy='omit')


In [ ]:
sns.scatterplot(x='age', y='thalach', hue='target', data=df, palette='coolwarm')
plt.title("Age vs Max Heart Rate by Target")


In [ ]:
fig = px.histogram(data_frame=df, x='age', color= 'sex')
fig.show()

In [ ]:
df['sex'].value_counts()

In [ ]:
sns.countplot(df,x='sex')

In [ ]:
male_count = 206
female_count = 97

total_count = male_count + female_count

# calculate percentages
male_percentage = (male_count/total_count)*100
female_percentages = (female_count/total_count)*100

# display the results
print(f'Male percentage in the data: {male_percentage:.2f}%')
print(f'Female percentage in the data : {female_percentages:.2f}%')

# Difference
difference_percentage = ((male_count - female_count)/female_count) * 100
print(f'Males are {difference_percentage:.2f}% more than female in the data.')


In [ ]:
df.groupby('sex')['age'].value_counts()

## Exploring CP (Chest Pain) column

In [ ]:
df['cp'].value_counts()

In [ ]:
sns.countplot(df,x='cp')

In [ ]:
sns.countplot(df, x='cp', hue= 'sex')

In [ ]:

fig = px.histogram(data_frame=df, x='age', color='cp')
fig.show()

### Let's explore the trestbps (resting blood pressure) column:

In [ ]:

df['trestbps'].describe()

In [ ]:
sns.histplot(data=df, x='trestbps', kde=True,)

plt.title('Resting Blood Pressure')
plt.xlabel('Pressure (mmHg)')
plt.ylabel('Count')

In [ ]:
sns.histplot(df, x='trestbps', kde=True, hue ='sex') 

<h2>Impute missing values using iterative imputer for selected columns.</h2>

columns are selected based on data types (floating data type)
because imputer only works with the floating data types.

selected columns are:
1. thal
2. ca

In [ ]:
imputer = SimpleImputer(strategy='most_frequent')
df['ca'] = imputer.fit_transform(df[['ca']]).ravel()
df['thal'] = imputer.fit_transform(df[['thal']]).ravel()

In [ ]:
# let's check again for missing values
(df.isnull().sum()).sort_values(ascending=False)

Now Missing values are imputed and there is no missing values in the columns....

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
X = df.drop(['target'], axis=1)
y = df['target']

<h3>Splitting the dataset into training and testing sets</h3>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

In [ ]:
scaler = StandardScaler() 

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])


<h2>MODEL DEVELOPMENT</h2>

<h3>1. Logistic Regression</h3>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, classification_report
import numpy as np

# Cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic Regression with different penalties
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet'],
    'C': np.logspace(-3, 3, 7),
    'solver': ['saga'],
    'l1_ratio': [0, 0.25, 0.5, 0.75, 1] 
}

logreg = LogisticRegression(max_iter=1000)

grid = GridSearchCV(logreg, param_grid, cv=cv, scoring='f1', n_jobs=-1, error_score='raise')
grid.fit(X_train_scaled, y_train)

cal_lgr = CalibratedClassifierCV(grid.best_estimator_, cv=cv)
cal_lgr.fit(X_train_scaled, y_train)

y_pred = cal_lgr.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

coef = grid.best_estimator_.coef_[0]
features = X_train_scaled.columns
importance_df = pd.DataFrame({'Feature': features, 'Coefficient': coef})
importance_df = importance_df.sort_values(by='Coefficient', key=abs, ascending=False)

sns.barplot(data=importance_df, x='Coefficient', y='Feature')
plt.title("Logistic Regression Feature Importance")
plt.show()


<h3>2. K-Nearest Neighbors (KNN)</h3>

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

knn_pipeline = Pipeline([
    ('scaler', StandardScaler()), 
    ('knn', KNeighborsClassifier())
])

param_grid_knn = {
    'knn__n_neighbors': list(range(3, 22, 2)), 
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan', 'minkowski']
}

cal_knn = GridSearchCV(knn_pipeline, param_grid_knn, cv=cv, scoring='f1', n_jobs=-1)
cal_knn.fit(X_train, y_train)

y_pred_knn = cal_knn.predict(X_test)
print("Best KNN params:", cal_knn.best_params_)
print(classification_report(y_test, y_pred_knn))

In [ ]:
results = pd.DataFrame(cal_knn.cv_results_)
scores_k = results[results['param_knn__metric'] == 'euclidean']

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
sns.lineplot(data=scores_k, x='param_knn__n_neighbors', y='mean_test_score', hue='param_knn__weights')
plt.title("KNN: F1-score vs K value (Euclidean)")
plt.ylabel("F1-score")
plt.show()


<h3>3. Support Vector Machine (SVM)</h3>

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True, class_weight='balanced'))
])

param_grid_svm = {
    'svm__kernel': ['linear', 'rbf', 'poly'],
    'svm__C': [0.1, 1, 10],
    'svm__gamma': ['scale', 'auto'],     # For RBF/poly
    'svm__degree': [2, 3]                # Only relevant for 'poly'
}

grid_svm = GridSearchCV(
    svm_pipeline, param_grid=param_grid_svm,
    cv=cv, scoring='f1', n_jobs=-1
)
grid_svm.fit(X_train, y_train)

cal_svm = CalibratedClassifierCV(estimator=grid_svm.best_estimator_, cv=cv)
cal_svm.fit(X_train, y_train)

y_pred_svm = cal_svm.predict(X_test)
print("Best SVM Params:", grid_svm.best_params_)
print(classification_report(y_test, y_pred_svm))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

X_test_scaled = grid_svm.best_estimator_.named_steps['scaler'].transform(X_test)

perm_importance = permutation_importance(cal_svm, X_test_scaled, y_test, n_repeats=20, random_state=42)

importances = pd.DataFrame({
    'feature': X_test.columns,
    'importance': perm_importance.importances_mean
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=importances.head(10), y='feature', x='importance')
plt.title("SVM Permutation Feature Importance")
plt.show()


<h3>4. Random Forest Classifier (RF) </h3>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 'log2']
}

rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)

grid_rf = GridSearchCV(
    rf_model, param_grid_rf, cv=cv,
    scoring='f1', n_jobs=-1
)
grid_rf.fit(X_train, y_train)

cal_rf = CalibratedClassifierCV(grid_rf.best_estimator_, cv=cv)
cal_rf.fit(X_train, y_train)

y_pred_rf = cal_rf.predict(X_test)
print("Best RF Params:", grid_rf.best_params_)
print(classification_report(y_test, y_pred_rf))


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

feat_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': cal_rf.estimator.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=feat_imp.head(10), y='feature', x='importance')
plt.title("Random Forest Feature Importance")
plt.show()


<h3>5. XGBoost Classifier</h3>

In [ ]:
from xgboost import XGBClassifier, plot_importance
from sklearn.model_selection import GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report

param_grid_xgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0]
}

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

grid_xgb = GridSearchCV(xgb, param_grid_xgb, cv=cv, scoring='f1', n_jobs=-1)
grid_xgb.fit(X_train, y_train)

cal_xgb = CalibratedClassifierCV(grid_xgb.best_estimator_, cv=cv)
cal_xgb.fit(X_train, y_train)

y_pred_xgb = cal_xgb.predict(X_test)
print("Best XGBoost Params:", grid_xgb.best_params_)
print(classification_report(y_test, y_pred_xgb))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plot_importance(grid_xgb.best_estimator_, importance_type='gain', max_num_features=10)
plt.title("XGBoost Feature Importance (Gain)")
plt.show()


<h3>6. Light GBM</h3>

In [ ]:
from lightgbm import LGBMClassifier, plot_importance
from sklearn.model_selection import GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report

param_grid_lgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5, 7],
    'num_leaves': [15, 31, 63]
}

lgb = LGBMClassifier(class_weight='balanced', random_state=42)

grid_lgb = GridSearchCV(lgb, param_grid_lgb, cv=cv, scoring='f1', n_jobs=-1)
grid_lgb.fit(X_train, y_train)

cal_lgb = CalibratedClassifierCV(grid_lgb.best_estimator_, cv=cv)
cal_lgb.fit(X_train, y_train)

y_pred_lgb = cal_lgb.predict(X_test)
print("Best LightGBM Params:", grid_lgb.best_params_)
print(classification_report(y_test, y_pred_lgb))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plot_importance(grid_lgb.best_estimator_, max_num_features=10)
plt.title("LightGBM Feature Importance")
plt.show()

<h2>Evaluation of Traditional Machine learning models </h2>

In [ ]:
models = {
    'Logistic Regression': cal_lgr,  
    'KNN': cal_knn,            
    'SVM': cal_svm,
    'Random Forest': cal_rf,
    'XGBoost': cal_xgb,
    'LightGBM': cal_lgb
}

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-score': f1_score(y_test, y_pred),
        'Best Params': getattr(model, 'best_params_', 'N/A') 
    }

def evaluate_all_models(models: dict, X_test, y_test):
    results = [evaluate_model(name, model, X_test, y_test) for name, model in models.items()]
    df_results = pd.DataFrame(results).sort_values(by='F1-score', ascending=False)
    
    print("Model Comparison:\n")
    print(df_results[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-score']])
    
    best_model_row = df_results.iloc[0]
    print(f"\nBest Model: {best_model_row['Model']}")
    
    return df_results


In [ ]:
df_results = evaluate_all_models(models, X_test, y_test)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curves for All Models')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

for name, model in models.items():
    plt.figure()
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)
    plt.title(f"Confusion Matrix - {name}")
    plt.grid(False)
    plt.show()


<h2>MLP model (Deep Learning)</h2>

In [ ]:
import torch 
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values.reshape(-1, 1), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values.reshape(-1, 1), dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

class HeartDiseasePredictor(nn.Module):
    def __init__(self, input_dim, dropout_rate=0.3):
        super(HeartDiseasePredictor, self).__init__()

        self.fc1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)

        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)

        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)

        self.fc4 = nn.Linear(64, 32)
        self.bn4 = nn.BatchNorm1d(32)

        self.out = nn.Linear(32, 1)

        self.dropout = nn.Dropout(dropout_rate)
        self.activation = nn.LeakyReLU(0.1)

    def forward(self, x):
        x = self.dropout(self.activation(self.bn1(self.fc1(x))))
        x = self.dropout(self.activation(self.bn2(self.fc2(x))))
        x = self.dropout(self.activation(self.bn3(self.fc3(x))))
        x = self.dropout(self.activation(self.bn4(self.fc4(x))))
        return self.out(x)


In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
from sklearn.metrics import f1_score, classification_report
from torch.utils.data import TensorDataset, DataLoader

model = HeartDiseasePredictor(input_dim=X_train.shape[1])
optimizer = optim.AdamW(model.parameters(), lr=0.001)

# Class imbalance handling
pos_weight = torch.tensor([len(y_train) / sum(y_train) - 1])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# Training loop
n_epochs = 50
best_val_loss = float('inf')
patience = 5
early_stop_counter = 0

train_losses, val_losses = [], []

for epoch in range(n_epochs):
    model.train()
    train_loss = 0.0

    for Xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    with torch.no_grad():
        val_logits = model(X_test_tensor)
        val_loss = criterion(val_logits, y_test_tensor).item()
        scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

    # Early Stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        if early_stop_counter >= patience:
            print("Early stopping triggered.")
            break


<h3>Evaluation Metrics</h3>

In [ ]:
model.load_state_dict(best_model_state)
model.eval()

with torch.no_grad():
    probs = torch.sigmoid(model(X_test_tensor)).numpy()

best_thresh, best_f1 = 0.5, 0
for t in np.arange(0.3, 0.7, 0.01):
    preds = (probs >= t).astype(int)
    f1 = f1_score(y_test, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
\
final_preds = (probs >= best_thresh).astype(int)
print(f"\nBest threshold: {best_thresh:.2f}, F1 Score: {best_f1:.4f}")
print(classification_report(y_test, final_preds))


In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss (MLP)")
plt.legend()
plt.grid()
plt.show()


In [ ]:
def evaluate_model(name, model, X_test, y_test, mlp=False, threshold=0.6):
    if mlp:
        model.eval()
        with torch.no_grad():
            X_tensor = torch.tensor(X_test.values, dtype=torch.float32)
            y_proba = torch.sigmoid(model(X_tensor)).numpy().flatten()
        y_pred = (y_proba >= threshold).astype(int)
    else:
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

    return {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-score': f1_score(y_test, y_pred)
    }


In [ ]:
all_models = {
    'Logistic Regression': cal_lgr,
    'KNN': cal_knn,
    'SVM': cal_svm,
    'Random Forest': cal_rf,
    'XGBoost': cal_xgb,
    'LightGBM': cal_lgb,
    'MLP (PyTorch)': model  # assuming final MLP loaded & threshold tuned
}

In [ ]:
results = []

# scikit-learn models
results.append(evaluate_model("Logistic Regression", cal_lgr, X_test, y_test))
results.append(evaluate_model("KNN", cal_knn, X_test, y_test))
results.append(evaluate_model("SVM", cal_svm, X_test, y_test))
results.append(evaluate_model("Random Forest", cal_rf, X_test, y_test))
results.append(evaluate_model("XGBoost", cal_xgb, X_test, y_test))
results.append(evaluate_model("LightGBM", cal_lgb, X_test, y_test))

results.append(evaluate_model("MLP (PyTorch)", model, X_test, y_test, mlp=True, threshold=0.65))

# Create results DataFrame
df_results = pd.DataFrame(results).sort_values(by='F1-score', ascending=False)
print(df_results)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(10, 7))

for name, m in all_models.items():
    if hasattr(m, "predict_proba"):
        y_scores = m.predict_proba(X_test)[:, 1]
    else:  # MLP case
        with torch.no_grad():
            y_scores = torch.sigmoid(m(torch.tensor(X_test.values, dtype=torch.float32))).numpy().flatten()
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc(fpr, tpr):.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid()
plt.show()

<h2>Ensemble Voting Classifier</h2>

In [ ]:
from sklearn.ensemble import VotingClassifier

voting = VotingClassifier(estimators=[
    ('rf', cal_rf),
    ('xgb', cal_xgb),
    ('lr', cal_lgr)
], voting='soft')

voting.fit(X_train, y_train)
y_pred = voting.predict(X_test)

print("Voting Classifier Performance:")
print(classification_report(y_test, y_pred))



In [ ]:
results = []

results.append(evaluate_model("Logistic Regression", cal_lgr, X_test, y_test))
results.append(evaluate_model("KNN", cal_knn, X_test, y_test))
results.append(evaluate_model("SVM", cal_svm, X_test, y_test))
results.append(evaluate_model("Random Forest", cal_rf, X_test, y_test))
results.append(evaluate_model("XGBoost", cal_xgb, X_test, y_test))
results.append(evaluate_model("LightGBM", cal_lgb, X_test, y_test))
results.append(evaluate_model("Ensemble", voting, X_test, y_test))

results.append(evaluate_model("MLP (PyTorch)", model, X_test, y_test, mlp=True, threshold=0.65))

df_results = pd.DataFrame(results).sort_values(by='F1-score', ascending=False)
print(df_results)
